# Chapter 4 &mdash; Grossly Abusing the Pumping Lemma

**Concept 20 of the Chapter 4 decomposition:** *Grossly Abusing the Pumping Lemma: the Language $L_{if}$*

For someone wielding a hammer every problem looks like a nail &mdash; but $L_{if}$ is not a nail.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-Abusing-The-Pumping-Lemma/Concept-Abusing-The-Pumping-Lemma.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


The lemma given here is a **rubber hammer**: it works often enough, but not always.
There is also a **wooden hammer** (the more general lemma, Concept 22) and a
**vanadium-steel hammer** (a *complete* pumping lemma, e.g. Jaffe's).

On $L_{if} = \{a^ib^jc^k : \text{if } i=3 \text{ then } j=k\}$, walking the cases shows
three of four break &mdash; and the fourth does not. **Only if you break them all have you
proved anything.**

## 2. Definitions

### The four cases, spelled out

In [ ]:
def in_Lif(s):
    i = len(s) - len(s.lstrip('a')); rest = s[i:]
    j = len(rest) - len(rest.lstrip('b')); k = len(rest) - j
    if rest[j:] != 'c'*k or s != 'a'*i + 'b'*j + 'c'*k: return False
    return (j == k) if i == 3 else True

def where(y, w):
    if set(y) <= {'a'}: return 'inside the a-block'
    if set(y) <= {'b'}: return 'inside the b-block'
    if set(y) <= {'a','b'}: return 'straddling a and b'
    if set(y) <= {'b','c'}: return 'straddling b and c'
    return 'other'

### Classify every split by where the pump lands

In [ ]:
def case_report(w, N, imax=3):
    out = {}
    for p in range(N+1):
        for q in range(p+1, min(N, len(w))+1):
            x, y, z = w[:p], w[p:q], w[q:]
            broken = any(not in_Lif(x + y*i + z) for i in range(imax+1))
            out.setdefault(where(y, w), []).append(broken)
    return out

## 3. Tests

Three cases break; the fourth survives.

In [ ]:
N = 4; w = 'aaa' + 'b'*N + 'c'*N
rep = case_report(w, N)
for k in sorted(rep):
    br = sum(rep[k]); tot = len(rep[k])
    print("%-22s %2d/%2d splits broken  %s" % (k, br, tot, "ALL" if br==tot else "<-- NOT ALL"))
assert any(sum(v) < len(v) for v in rep.values()), "some case must survive"

The surviving case, and why it survives.

In [ ]:
survivors = [(p,q) for p in range(N+1) for q in range(p+1, min(N,len(w))+1)
             if all(in_Lif(w[:p] + w[p:q]*i + w[q:]) for i in range(4))]
x,y,z = w[:survivors[0][0]], w[survivors[0][0]:survivors[0][1]], w[survivors[0][1]:]
print("surviving split : x=%r y=%r z=%r" % (x,y,z))
print("y lies", where(y, w))
print("\nPumping a's moves the a-count off 3, so 'if i=3 then j=k' is vacuously true.")

So the proof attempt fails &mdash; and cherry-picking the other three would be fraud.

In [ ]:
print("cases broken  : 3 of 4")
print("proof status  : FAILED -- we may not conclude non-regularity from this")
print()
print("Analogy: proving x != x^2 by checking x = 2, 3, 5 and ignoring x = 0, 1.")

## 4. Exercises


1. Which of the four cases do you find least obvious? Work it by hand.
2. Try $w = aaa\,b^Nc^{N+1}$ instead. Does that help?
3. The book offers two ways out. Name them. (Concepts 21, 22.)

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter4/Concept-Abusing-The-Pumping-Lemma')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')